In [1]:
import os
import pathlib
import json
import time
import pandas as pd
from IPython.display import display, Markdown

from google import genai
from google.genai import types, errors
pd.set_option('display.max_columns', 100)
from pathlib import Path

key = Path("key.txt").read_text().strip()

In [2]:
prompt = """
You are a specialized assistant for analyzing municipal zoning and development codes and outputting structured data with one row per distinct zoning district. You work from complete municipal code PDFs (or direct links) and must process the entire document—no summaries or partial reads. If the code is split across multiple PDFs or volumes, halt and state exactly which additional files are required before proceeding.



When files or links are provided, read and analyze the entire code end-to-end, including all text, tables, appendices, figures, and footnotes. Identify all base zoning districts (e.g., R-1, R-4, NB, CBD) and any overlay zones. Treat each unique zoning district as a separate row. Only consolidate zones if their names are truly synonymous and their standards are identical.



For every zone, extract these development standards:

- Juris	Jurisdiction (city or town name like Bellevue)

- Zone	zone name (like R-1, or R-4-C)

- Juris_zn unique id - (Juris plus zone)

 -Definition	text description of zone

- MinDU (minimum dwelling units per acre)

- MaxDU (maximum dwelling units per acre)

- MinFAR (minimum floor area ratio)

- MaxFAR (maximum floor area ratio)

- MaxHt (maximum building height; use the highest permitted with bonuses or geographic exceptions)

- LC (maximum zone coverage percentage — the share of the site area that may be covered by buildings/structures, using the code’s own definition)


If an overlay supersedes a base zone value, use the overlay standard in the corresponding fields.



Extract values precisely from the code. Never fabricate or generalize values. Do not substitute values from other zones. Do not estimate or fill with similar-looking standards. For each metric, return only what is explicitly stated in the code section or table.



For conditional, location-based, or multi-tiered standards (height, FAR, DU, LC, etc.):

- When multiple values exist for any metric based on geography, lot size, frontage, housing type, or incentive category, extract **the maximum value applicable within the defined conditions of that row type**.

- In the **bonus-excluded row**, compute the maximum of all values **excluding any bonus-derived allowances**.

- In the **bonus-included row**, compute the maximum of all values **including the effect of all applicable bonuses and incentives**.

- Do not average, round, or substitute values. Be precise.

- Do not stack bonuses unless the code explicitly permits stacking.

- Never treat conditional site-based allowances (e.g., 'south of 268th St') as bonuses unless they are explicitly framed as incentive-based mechanisms.



Assign use flags strictly from the code’s purpose statements and permitted uses:

- Bonus_avail = 'Y' if bonuses/incentives are available in any form for the zone; otherwise ''.

- Res_Use, Comm_Use, Office_Use, Indust_Use, Mixed_Use, Public_Use, PUD_Use set to 'Y' only if those uses are permitted as primary uses. Mixed_Use = 'Y' only when both residential and commercial are permitted as primary uses.



Column population rule (enforced): Only populate *_Res when Res_Use = 'Y'; *_Comm when Comm_Use = 'Y'; *_Office when Office_Use = 'Y'; *_Indust when Indust_Use = 'Y'; *_Mixed when Mixed_Use = 'Y'. Leave all other use-suffix fields null. When the code specifies different standards by use, extract the values exactly as written for each use.

Example: If residential and commercial have different FARs, populate the values accordingly in _Res and _Comm.




**Bonus row logic**: If bonuses are available for a zone, always create two rows:

- One row reflecting development standards excluding all bonuses (Bonus_included = '').

- One row reflecting maximum development standards with all applicable bonuses included (Bonus_included = 'Y').

- In both rows, set Bonus_avail = 'Y'.

- For each field, only include bonus-enhanced maximums in the Bonus_included = 'Y' row. In the Bonus_included = '' row, exclude bonus-enhanced values.



**FAR formatting**: Convert ratios like 2.5/1 or 3/1 to floats 2.5 or 3. FAR values must be numeric (ints or floats).



**Default output format**: Produce a JSON array of objects, one per zone, using exactly these keys in this exact order and no extras. Every object must include all keys; for unavailable values use null; for flags use 'Y' or ''. For numeric metrics, use integers where possible, otherwise floats; never include units.



["Juris","Zone","Juris_zn","Definition","Bonus_avail","Bonus_included","Res_Use","Comm_Use","Office_Use","Indust_Use","Mixed_Use","Public_Use","PUD_Use","ResDU_lot","MinDU_Res","MinDU_Comm","MinDU_Office","MinDU_Indust","MinDU_Mixed","MaxDU_Res","MaxDU_Comm","MaxDU_Office","MaxDU_Indust","MaxDU_Mixed","MinFAR_Res","MinFAR_Comm","MinFAR_Office","MinFAR_Indust","MinFAR_Mixed","MaxFAR_Res","MaxFAR_Comm","MaxFAR_Office","MaxFAR_Indust","MaxFAR_Mixed","MaxHt_Res","MaxHt_Comm","MaxHt_Office","MaxHt_Indust","MaxHt_Mixed","LC_Res","LC_Comm","LC_Office","LC_Indust","LC_Mixed","rural","ADU notes"]



**CSV option** (only if explicitly requested): Output a fenced `csv` code block using the exact header above (comma-separated). Ensure every row has exactly the same number of fields as the header. No prose before/after unless the user explicitly requests notes.



**Validation before output** (hard stop rules):

- JSON: ensure each object contains every key exactly once; if a key is missing for a zone, add it with null. No extra keys.

- CSV: ensure each row has exactly the same number of columns as the header. If needed, insert empty trailing fields to match the header count (never drop real data).

- Enforce the column population rule: if a use flag is not 'Y', all corresponding *_suffix fields must be null (JSON) or blank (CSV).



**Error handling**: If parsing fails for any zone or table, identify the exact section/table reference and briefly state why (e.g., malformed data, ambiguous headers, conflicting values). Do not proceed with incomplete or silently skipped output—stop and clearly state what needs to be resolved (including which missing files/volumes are referenced by the code).



**Interaction and style**: Never fabricate data. When a file is provided, immediately process the entire document and output the data in the default JSON format. Do not ask questions or seek confirmations. Extract the jurisdiction name directly from the document. Only include a Notes section if the user explicitly requests it. Keep tone neutral and concise; no meta commentary.



**Tools and retrieval**: When links are provided, fetch the PDFs and use a PDF-friendly reading mode (including screenshot capture for tables when needed). If the code references additional volumes or separate chapters (e.g., definitions, use tables, overlays), stop and indicate precisely which parts are required to continue. Do all work synchronously in the current response; do not promise future updates."""


In [3]:
# -------------------------
# Configuration
# -------------------------
PDF_FOLDER  = pathlib.Path("Cities")
MODEL_NAME  = "models/gemini-2.5-pro"

OUTPUT_CSV      = "all_zoning_data_batch.csv"
OUTPUT_JSON     = "all_zoning_data_batch.json"
RAW_LOG_FILE    = "all_zoning_batch_raw.jsonl"  


GEMINI_API_KEY = key
ZONING_PROMPT  = prompt

DESIRED_COLUMNS = [
    "Juris","Zone","Juris_zn","Definition","Bonus_avail","Bonus_included",
    "Res_Use","Comm_Use","Office_Use","Indust_Use","Mixed_Use","Public_Use","PUD_Use",
    "ResDU_lot","MinDU_Res","MinDU_Comm","MinDU_Office","MinDU_Indust","MinDU_Mixed",
    "MaxDU_Res","MaxDU_Comm","MaxDU_Office","MaxDU_Indust","MaxDU_Mixed",
    "MinFAR_Res","MinFAR_Comm","MinFAR_Office","MinFAR_Indust","MinFAR_Mixed",
    "MaxFAR_Res","MaxFAR_Comm","MaxFAR_Office","MaxFAR_Indust","MaxFAR_Mixed",
    "MaxHt_Res","MaxHt_Comm","MaxHt_Office","MaxHt_Indust","MaxHt_Mixed",
    "LC_Res","LC_Comm","LC_Office","LC_Indust","LC_Mixed","rural","ADU notes"
]

# How often to poll the batch job (seconds)
POLL_SECONDS = 30`

def run_batch_job():
    display(Markdown("##  Starting Batch Zoning Job"))

    if not GEMINI_API_KEY:
        print("ERROR: Set GEMINI_API_KEY / key variable.")
        return

    if not PDF_FOLDER.exists() or not PDF_FOLDER.is_dir():
        print(f"ERROR: Folder not found: {PDF_FOLDER.resolve()}")
        return

    pdf_paths = sorted(PDF_FOLDER.glob("*.pdf"))
    if not pdf_paths:
        print(f"No PDF files found in {PDF_FOLDER.resolve()}")
        return

    client = genai.Client(api_key=GEMINI_API_KEY)

    try:
        print(f"Found {len(pdf_paths)} PDF files to process in batch.")
        uploaded_files = []

        # -------------------------
        # 1) Upload all PDFs
        # -------------------------
        for i, pdf_path in enumerate(pdf_paths, start=1):
            print(f"Uploading {i}/{len(pdf_paths)}: {pdf_path.name}")
            try:
                uf = client.files.upload(
                    file=str(pdf_path),
                    config=types.UploadFileConfig(
                        display_name=pdf_path.name,
                        mime_type="application/pdf",
                    ),
                )
                uploaded_files.append((pdf_path.name, uf))
            except Exception as e:
                print(f"  -> ERROR uploading {pdf_path.name}: {e}")

        if not uploaded_files:
            print("No files uploaded successfully; aborting.")
            return

        # -------------------------
        # 2) Build inline requests (one GenerateContentRequest per PDF)
        # -------------------------
        inline_requests = []
        for pdf_name, uf in uploaded_files:
            # Each element is a GenerateContentRequest as a dict
            req = {
                "contents": [{
                    "role": "user",
                    "parts": [
                        {"text": ZONING_PROMPT},
                        {
                            "fileData": {
                                "fileUri": uf.uri,   # reference uploaded PDF
                                "mimeType": "application/pdf",
                            }
                        },
                    ],
                }],
                "config": {
                    # Ask for machine-parseable JSON
                    "response_mime_type": "application/json",
                },
                "metadata": {"key": pdf_name},
            }
            inline_requests.append(req)

        # -------------------------
        # 3) Create batch job
        # -------------------------
        print("\nCreating batch job...")
        inline_batch_job = client.batches.create(
            model=MODEL_NAME,
            src=inline_requests,
            config={
                "display_name": "zoning-codes-inline-batch",
            },
        )

        job_name = inline_batch_job.name
        print(f"Created batch job: {job_name}")

        # -------------------------
        # 4) Poll until the job completes
        # -------------------------
        completed_states = {
            "JOB_STATE_SUCCEEDED",
            "JOB_STATE_FAILED",
            "JOB_STATE_CANCELLED",
            "JOB_STATE_EXPIRED",
        }

        print(f"\nPolling status for job: {job_name}")
        while True:
            batch_job = client.batches.get(name=job_name)
            state = batch_job.state.name
            print(f"  Current state: {state}")
            if state in completed_states:
                break
            time.sleep(POLL_SECONDS)

        print(f"\nJob finished with state: {batch_job.state.name}")
        if batch_job.state.name != "JOB_STATE_SUCCEEDED":
            print("Batch did not succeed; nothing to parse.")
            if batch_job.error:
                print(f"Error: {batch_job.error}")
            return

        # -------------------------
        # 5) Collect inline responses
        # -------------------------
        dest = getattr(batch_job, "dest", None)
        inlined = getattr(dest, "inlined_responses", None) if dest else None

        if not inlined:
            print("No inline responses found in batch job.")
            return

        all_rows = []
        print(f"\nFound {len(inlined)} inline responses.")
        print(f"Logging raw responses to: {RAW_LOG_FILE}")

        # Open raw log file once, write one JSON line per response
        with open(RAW_LOG_FILE, "w", encoding="utf-8") as log_f:

            for idx, inline_response in enumerate(inlined, start=1):
                print(f"\n--- Parsing response {idx} ---")

                # Map back to source file via index
                src_file_name = uploaded_files[idx-1][0] if idx-1 < len(uploaded_files) else None

                resp = getattr(inline_response, "response", None)
                err  = getattr(inline_response, "error", None)

                # --- Build raw log entry before any parsing ---
                raw_log_entry = {
                    "index": idx,
                    "source_file": src_file_name,
                }

                if resp:
                    # Try to get structured JSON and raw text
                    parsed_for_log = getattr(resp, "parsed", None)
                    text_for_log   = getattr(resp, "text", None)

                    # parsed may not be JSON-serializable as-is, so we convert carefully
                    if parsed_for_log is not None:
                        try:
                            # if it's already a dict/list of simple types, this just works
                            json.dumps(parsed_for_log)
                            raw_log_entry["parsed"] = parsed_for_log
                        except TypeError:
                            # fallback: string repr
                            raw_log_entry["parsed"] = repr(parsed_for_log)

                    if text_for_log is not None:
                        raw_log_entry["text"] = text_for_log
                else:
                    # No response, only error
                    raw_log_entry["error"] = {
                        "code": getattr(err, "code", None),
                        "message": getattr(err, "message", None) or repr(err),
                    }

                # Write raw log line
                log_f.write(json.dumps(raw_log_entry, ensure_ascii=False) + "\n")

                # normal parsing for DataFrame
                if not resp:
                    print(f"  -> No response; error={err}")
                    continue

                # Try .parsed first (JSON mode), then .text
                parsed = getattr(resp, "parsed", None)
                if parsed is None:
                    txt = getattr(resp, "text", None)
                    if txt:
                        try:
                            parsed = json.loads(txt)
                        except Exception:
                            # As a last resort, try to extract JSON array substring
                            start = txt.find("[")
                            end = txt.rfind("]")
                            try:
                                if start != -1 and end != -1:
                                    parsed = json.loads(txt[start:end+1])
                            except Exception:
                                parsed = None

                if not parsed:
                    print("  -> FAILED: Could not parse JSON payload for this response.")
                    continue

                # Normalize to list-of-dicts
                if isinstance(parsed, dict):
                    parsed = [parsed]
                if not isinstance(parsed, list):
                    print("  -> FAILED: Parsed JSON is not a list; skipping.")
                    continue

                all_rows.extend(parsed)
                print(f"  -> Success: Extracted {len(parsed)} rows from response {idx}.")

        if not all_rows:
            print("\nBatch completed, but no data was successfully extracted.")
            return

        print(f"\nTotal rows extracted from batch: {len(all_rows)}")

        # -------------------------
        # 6) Build DataFrame and save CSV/JSON
        # -------------------------
        df = pd.DataFrame(all_rows)

        for col in DESIRED_COLUMNS:
            if col not in df.columns:
                df[col] = pd.NA

        df = df[DESIRED_COLUMNS]

        df.to_csv(OUTPUT_CSV, index=False)
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

        print(f"\nSuccessfully saved:")
        print(f"  -> CSV:  {OUTPUT_CSV}")
        print(f"  -> JSON: {OUTPUT_JSON}")
        print(f"  -> RAW:  {RAW_LOG_FILE}\n")
        display(df.head())

    finally:
        #clean up remote files now that job is done
        try:
            for _, uf in uploaded_files:
                try:
                    client.files.delete(name=uf.name)
                except Exception as del_e:
                    print(f"Cleanup warning for {uf.name}: {del_e}")
        except Exception:
            pass

        try:
            client.close()
        except Exception:
            pass

run_batch_job()


## 🚀 Starting Batch Zoning Job (google-genai Batch API)

Found 80 PDF files to process in batch.
Uploading 1/80: Algona.pdf
Uploading 2/80: Arlington.pdf
Uploading 3/80: Auburn.pdf
Uploading 4/80: BainbridgeIsland.pdf
Uploading 5/80: Beaux Arts Village.pdf
Uploading 6/80: Bellevue.pdf
Uploading 7/80: Black Diamond.pdf
Uploading 8/80: Bonney Lake.pdf
Uploading 9/80: Bothell.pdf
Uploading 10/80: Brementon.pdf
Uploading 11/80: Brier.pdf
Uploading 12/80: Buckley.pdf
Uploading 13/80: Burien.pdf
Uploading 14/80: Carbonado.pdf
Uploading 15/80: Carnation.pdf
Uploading 16/80: ClydeHill.pdf
Uploading 17/80: Covington.pdf
Uploading 18/80: Darrington.pdf
Uploading 19/80: Des Moines.pdf
Uploading 20/80: DuPont.pdf
Uploading 21/80: Duvall.pdf
Uploading 22/80: Eatonville.pdf
Uploading 23/80: Edgewood.pdf
Uploading 24/80: Edmonds.pdf
Uploading 25/80: Enumclaw.pdf
Uploading 26/80: Everett.pdf
Uploading 27/80: FederalWay.pdf
Uploading 28/80: Fife.pdf
Uploading 29/80: Fircrest.pdf
Uploading 30/80: Gig Harbor.pdf
Uploading 31/80: Gold Bar.pdf
Uploading 32/80: G

,Juris,Zone,Juris_zn,Definition,Bonus_avail,Bonus_included,Res_Use,Comm_Use,Office_Use,Indust_Use,Mixed_Use,Public_Use,PUD_Use,ResDU_lot,MinDU_Res,MinDU_Comm,MinDU_Office,MinDU_Indust,MinDU_Mixed,MaxDU_Res,MaxDU_Comm,MaxDU_Office,MaxDU_Indust,MaxDU_Mixed,MinFAR_Res,MinFAR_Comm,MinFAR_Office,MinFAR_Indust,MinFAR_Mixed,MaxFAR_Res,MaxFAR_Comm,MaxFAR_Office,MaxFAR_Indust,MaxFAR_Mixed,MaxHt_Res,MaxHt_Comm,MaxHt_Office,MaxHt_Indust,MaxHt_Mixed,LC_Res,LC_Comm,LC_Office,LC_Indust,LC_Mixed,rural,ADU notes
0,Algona,R-L Low Density Residential,R-L,The R-L low density residential district is in...,,,Y,,,,,Y,,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,NaN,None,"Up to two ADUs permitted per lot, do not count..."
1,Algona,R-M Medium Density Residential,R-M,The R-M medium density residential district is...,Y,,Y,Y,Y,,Y,Y,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.0,NaN,NaN,NaN,NaN,65.0,None,"Up to two ADUs permitted per lot, do not count..."
2,Algona,R-M Medium Density Residential,R-M,The R-M medium density residential district is...,Y,Y,Y,Y,Y,,Y,Y,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.0,NaN,NaN,NaN,NaN,65.0,None,"Up to two ADUs permitted per lot, do not count..."
3,Algona,C-1 Mixed Use Commercial,C-1,The C-1 mixed use commercial district is inten...,Y,,Y,Y,Y,,Y,Y,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.0,NaN,NaN,NaN,NaN,75.0,None,"Up to two ADUs permitted per lot, do not count..."
4,Algona,C-1 Mixed Use Commercial,C-1,The C-1 mixed use commercial district is inten...,Y,Y,Y,Y,Y,,Y,Y,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.5,NaN,NaN,NaN,NaN,75.0,None,"Up to two ADUs permitted per lot, do not count..."
